In [1]:
import os
import scanpy as sc
import pandas as pd

h5ad_path = os.path.expanduser("~/FYP_Regeneration/data/axolotl/processed/axolotl_processed.h5ad")
adata = sc.read_h5ad(h5ad_path)

print("Cell type categories (from the original study):")
print(adata.obs["celltype"].value_counts())

print("\nTimepoint categories:")
print(adata.obs["timepoint"].value_counts())

print("\nCell type x Timepoint breakdown:")
print(pd.crosstab(adata.obs["celltype"], adata.obs["timepoint"]))

print("\nCell type x your Leiden cluster breakdown:")
print(pd.crosstab(adata.obs["celltype"], adata.obs["leiden"]))

Cell type categories (from the original study):
celltype
5dpa            167
11dpa           163
18dpa           135
8dpa            121
Stage44         121
3dpa            108
fCTI             95
Stage28          82
fCTIII           80
Stage40          76
fCTIV            21
fCTV             20
tendon           13
periskeletal      7
Name: count, dtype: int64

Timepoint categories:
timepoint
5dpa       167
11dpa      163
18dpa      135
1apa       128
Stage44    121
8dpa       121
0dpa       108
3dpa       108
Stage28     82
Stage40     76
Name: count, dtype: int64

Cell type x Timepoint breakdown:
timepoint     0dpa  1apa  3dpa  5dpa  8dpa  11dpa  18dpa  Stage28  Stage40  \
celltype                                                                     
3dpa             0     0   108     0     0      0      0        0        0   
5dpa             0     0     0   167     0      0      0        0        0   
8dpa             0     0     0     0   121      0      0        0        0   
11dp

In [2]:
import pandas as pd
import os

tables_dir = os.path.expanduser("~/FYP_Regeneration/results/tables/")
targets = ["SP6", "SP8", "FGF8"]

axolotl_all = pd.read_csv(os.path.join(tables_dir, "axolotl_de_markers_all.csv"))
axolotl_hit = axolotl_all[axolotl_all["gene"].str.upper().isin(targets)]
print("=== Axolotl (all clusters, unfiltered) ===")
print(axolotl_hit[["cluster", "gene", "logfoldchanges", "pvals_adj"]].sort_values("gene"))

mouse_all = pd.read_csv(os.path.join(tables_dir, "mouse_de_genes_all.csv"))
mouse_hit = mouse_all[mouse_all["gene"].str.upper().isin(targets)]
print("\n=== Mouse (regenerative vs non-regenerative) ===")
print(mouse_hit[["gene", "log2FoldChange", "padj"]])

human_all = pd.read_csv(os.path.join(tables_dir, "human_fingertip_curated_reference.csv"))
human_hit = human_all[human_all["gene_symbol"].str.upper().isin(targets)]
print("\n=== Human (curated fingertip proteome) ===")
print(human_hit)

if axolotl_hit.empty and mouse_hit.empty and human_hit.empty:
    print("\nNone detected in any species' dataset — report this as a genuine "
          "negative result (platform/annotation limitation), not an error.")

=== Axolotl (all clusters, unfiltered) ===
        cluster  gene  logfoldchanges     pvals_adj
2813          0  FGF8        0.320980  9.984859e-01
108248        7  FGF8      -27.337585  9.110136e-01
95225         6  FGF8       -5.102118  1.246734e-02
69823         5  FGF8        2.035170  1.990814e-08
56436         4  FGF8        2.016887  1.487760e-05
52192         3  FGF8       -4.573446  3.714190e-04
41195         2  FGF8       -2.121708  1.726943e-01
123624        8  FGF8       -1.865516  6.041028e-01
27274         1  FGF8       -1.794403  3.904682e-01
78843         5   SP6       -0.383646  9.989843e-01
45111         3   SP6      -22.599707  9.860509e-01
120514        8   SP6        2.961243  9.120410e-01
5014          0   SP6      -22.536652  9.984859e-01
64447         4   SP6        0.134761  9.995085e-01
22676         1   SP6        0.271158  9.961071e-01
101524        7   SP6      -22.326910  9.936166e-01
33366         2   SP6        1.506891  1.000000e+00
86255         6   SP6

In [3]:
import scanpy as sc
import pandas as pd
import os

h5ad_path = os.path.expanduser("~/FYP_Regeneration/data/axolotl/processed/axolotl_processed.h5ad")
adata = sc.read_h5ad(h5ad_path)

# Check whether epidermal cells are even present in this table
print("Krt5 (epidermal marker) present in this dataset:",
      "KRT5" in adata.var_names.str.upper().values)

# Map timepoint -> regeneration status
regen_map = {
    "0dpa": "Uninjured",
    "1apa": "Regenerating", "3dpa": "Regenerating", "5dpa": "Regenerating",
    "8dpa": "Regenerating", "11dpa": "Regenerating", "18dpa": "Regenerating",
    "Stage28": "Developmental", "Stage40": "Developmental", "Stage44": "Developmental",
}
adata.obs["regen_status"] = adata.obs["timepoint"].map(regen_map)
print(adata.obs["regen_status"].value_counts())

# Drop developmental cells for this specific test — different biological process
adata_regen = adata[adata.obs["regen_status"].isin(["Uninjured", "Regenerating"])].copy()
print(f"\nCells in DE test: {adata_regen.obs['regen_status'].value_counts().to_dict()}")

sc.tl.rank_genes_groups(
    adata_regen, "regen_status",
    groups=["Regenerating"], reference="Uninjured",
    method="wilcoxon"
)

result = adata_regen.uns["rank_genes_groups"]
de_df = pd.DataFrame({
    "gene": result["names"]["Regenerating"],
    "logfoldchanges": result["logfoldchanges"]["Regenerating"],
    "pvals_adj": result["pvals_adj"]["Regenerating"],
})

# Same threshold as mouse (padj<0.05, |log2FC|>1) — fair cross-species comparison now
sig_regen_df = de_df[(de_df["pvals_adj"] < 0.05) & (de_df["logfoldchanges"].abs() > 1.0)]

tables_dir = os.path.expanduser("~/FYP_Regeneration/results/tables/")
de_df.to_csv(os.path.join(tables_dir, "axolotl_regen_vs_uninjured_all.csv"), index=False)
sig_regen_df.to_csv(os.path.join(tables_dir, "axolotl_regen_vs_uninjured_sig.csv"), index=False)

print(f"\nSignificant regeneration-associated genes: {len(sig_regen_df)} "
      f"(previously 8,532 using the old generic-cluster method)")
print(sig_regen_df.head(15))

print("\nSP6/SP8/FGF8 in this proper regeneration contrast:")
print(de_df[de_df["gene"].str.upper().isin(["SP6", "SP8", "FGF8"])])

Krt5 (epidermal marker) present in this dataset: True
regen_status
Regenerating     822
Developmental    279
Uninjured        108
Name: count, dtype: int64

Cells in DE test: {'Regenerating': 822, 'Uninjured': 108}

Significant regeneration-associated genes: 1753 (previously 8,532 using the old generic-cluster method)
        gene  logfoldchanges     pvals_adj
0    KAZALD2        4.092959  4.242684e-33
1      TGFBI        2.769974  5.227544e-33
2       NREP        4.482582  1.894276e-30
3       NASP        2.759443  3.851120e-25
4       SDC1        3.443279  8.382259e-25
5      STMN1        2.498775  8.382259e-25
6     RRM2.1        3.485763  1.134809e-23
7    COL11A1        2.812317  2.309321e-23
8        TNC        2.212553  7.557379e-22
9     SRSF1A        2.668954  8.267497e-22
10  AW551984        4.036623  5.731329e-18
11     UBE2C        3.377886  7.637587e-18
12     CDCA8        3.074258  2.092994e-17
13     CSTF1        2.306008  3.314662e-17
14      CDK1        4.460970  4.764

In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pybiomart import Server

tables_dir = os.path.expanduser("~/FYP_Regeneration/results/tables/")
fig_dir = os.path.expanduser("~/FYP_Regeneration/results/figures/cross_species/")
os.makedirs(fig_dir, exist_ok=True)

# 1. Load the NEW axolotl list (regeneration vs uninjured, not generic clusters) + same mouse/human as before
axolotl_df = pd.read_csv(os.path.join(tables_dir, "axolotl_regen_vs_uninjured_sig.csv"))
mouse_df = pd.read_csv(os.path.join(tables_dir, "mouse_de_genes_sig.csv"))
human_df = pd.read_csv(os.path.join(tables_dir, "human_fingertip_curated_reference.csv"))

raw_axolotl = set(axolotl_df["gene"].dropna().astype(str).str.upper())
raw_mouse = set(mouse_df["gene"].dropna().astype(str).str.upper())
raw_human = set(human_df["gene_symbol"].dropna().astype(str).str.upper())

print(f"Gene set sizes: axolotl(regen-specific)={len(raw_axolotl)}, mouse={len(raw_mouse)}, human={len(raw_human)}")

# 2. Mouse -> human ortholog mapping (same BioMart step as before)
server = Server(host="http://www.ensembl.org")
mouse_dataset = server.marts["ENSEMBL_MART_ENSEMBL"].datasets["mmusculus_gene_ensembl"]
homology = mouse_dataset.query(attributes=["external_gene_name", "hsapiens_homolog_associated_gene_name"])
homology.columns = ["mouse_symbol", "human_ortholog"]
homology = homology.dropna()
homology["mouse_symbol"] = homology["mouse_symbol"].str.upper()
homology["human_ortholog"] = homology["human_ortholog"].str.upper()
mouse_to_human = dict(zip(homology["mouse_symbol"], homology["human_ortholog"]))

mouse_ortho = set(mouse_to_human[g] for g in raw_mouse if g in mouse_to_human)
axolotl_ortho = raw_axolotl
human_ortho = raw_human

# 3. Real overlaps
axolotl_mouse = axolotl_ortho & mouse_ortho
axolotl_human = axolotl_ortho & human_ortho
mouse_human = mouse_ortho & human_ortho
tri_conserved = axolotl_ortho & mouse_ortho & human_ortho

print(f"\nCross-Species Overlaps (regen-specific axolotl list):")
print(f"- Axolotl & Mouse: {len(axolotl_mouse)} -> {sorted(axolotl_mouse)}")
print(f"- Axolotl & Human: {len(axolotl_human)} -> {sorted(axolotl_human)}")
print(f"- Mouse & Human: {len(mouse_human)}")
print(f"- Tri-Species Conserved: {len(tri_conserved)} -> {sorted(tri_conserved)}")

# 4. Save as v2 — keep your original files untouched for comparison in the write-up
all_genes = sorted(axolotl_ortho | mouse_ortho | human_ortho)
rows = []
for gene in all_genes:
    in_ax, in_ms, in_hu = gene in axolotl_ortho, gene in mouse_ortho, gene in human_ortho
    rows.append({
        "Ortholog_Gene_Symbol": gene, "Axolotl_scRNA": in_ax,
        "Mouse_BulkRNA": in_ms, "Human_Proteomics": in_hu,
        "Conservation_Score": sum([in_ax, in_ms, in_hu]),
    })
integrated_df = pd.DataFrame(rows)
conserved_df = integrated_df[integrated_df["Conservation_Score"] >= 2].sort_values("Conservation_Score", ascending=False)

integrated_df.to_csv(os.path.join(tables_dir, "tri_species_orthologs_all_v2.csv"), index=False)
conserved_df.to_csv(os.path.join(tables_dir, "tri_species_conserved_markers_v2.csv"), index=False)
print(f"\nSaved v2 results. Conservation_Score>=2 gene count: {len(conserved_df)}")

Gene set sizes: axolotl(regen-specific)=1753, mouse=196, human=60

Cross-Species Overlaps (regen-specific axolotl list):
- Axolotl & Mouse: 16 -> ['ABAT', 'ADTRP', 'BAMBI', 'C1QTNF2', 'COL14A1', 'HTRA4', 'IGFBP5', 'IL11', 'ITGBL1', 'LEF1', 'MATN4', 'MGST1', 'MSI1', 'MSX2', 'SOX11', 'THBS4']
- Axolotl & Human: 3 -> ['CAT', 'PRDX2', 'SERPINA1']
- Mouse & Human: 0
- Tri-Species Conserved: 0 -> []

Saved v2 results. Conservation_Score>=2 gene count: 19


In [5]:
import os
import pandas as pd
from scipy.stats import hypergeom
from pybiomart import Server

tables_dir = os.path.expanduser("~/FYP_Regeneration/results/tables/")

ax_all = pd.read_csv(os.path.join(tables_dir, "axolotl_regen_vs_uninjured_all.csv"))
ax_sig = pd.read_csv(os.path.join(tables_dir, "axolotl_regen_vs_uninjured_sig.csv"))
ms_all = pd.read_csv(os.path.join(tables_dir, "mouse_de_genes_all.csv"))
ms_sig = pd.read_csv(os.path.join(tables_dir, "mouse_de_genes_sig.csv"))

# mouse -> human mapping (same BioMart query as before)
server = Server(host="http://www.ensembl.org")
ds = server.marts["ENSEMBL_MART_ENSEMBL"].datasets["mmusculus_gene_ensembl"]
hom = ds.query(attributes=["external_gene_name", "hsapiens_homolog_associated_gene_name"]).dropna()
hom.columns = ["m", "h"]
m2h = dict(zip(hom["m"].str.upper(), hom["h"].str.upper()))

up = lambda s: set(s.dropna().astype(str).str.upper())
ax_universe, ax_sig_set = up(ax_all["gene"]), up(ax_sig["gene"])
ms_universe = {m2h[g] for g in up(ms_all["gene"]) if g in m2h}
ms_sig_set = {m2h[g] for g in up(ms_sig["gene"]) if g in m2h}

# Only genes measurable in BOTH species can possibly overlap
U = ax_universe & ms_universe
K = len(ax_sig_set & U)             # axolotl significant, within shared universe
n = len(ms_sig_set & U)             # mouse significant, within shared universe
k = len(ax_sig_set & ms_sig_set & U)

expected = K * n / len(U)
p = hypergeom.sf(k - 1, len(U), K, n)

print(f"Shared measurable genes (universe): {len(U)}")
print(f"Axolotl significant in universe: {K} | Mouse significant in universe: {n}")
print(f"Observed overlap: {k} | Expected by chance: {expected:.1f}")
print(f"Enrichment: {k/expected:.2f}x | Hypergeometric p-value: {p:.3g}")

Shared measurable genes (universe): 9740
Axolotl significant in universe: 1444 | Mouse significant in universe: 91
Observed overlap: 16 | Expected by chance: 13.5
Enrichment: 1.19x | Hypergeometric p-value: 0.269


In [6]:
import os
import pandas as pd
import gseapy as gp
from pybiomart import Server

tables_dir = os.path.expanduser("~/FYP_Regeneration/results/tables/")

ax = pd.read_csv(os.path.join(tables_dir, "axolotl_regen_vs_uninjured_sig.csv"))
ms = pd.read_csv(os.path.join(tables_dir, "mouse_de_genes_sig.csv"))
hu = pd.read_csv(os.path.join(tables_dir, "human_fingertip_curated_reference.csv"))

server = Server(host="http://www.ensembl.org")
ds = server.marts["ENSEMBL_MART_ENSEMBL"].datasets["mmusculus_gene_ensembl"]
hom = ds.query(attributes=["external_gene_name", "hsapiens_homolog_associated_gene_name"]).dropna()
hom.columns = ["m", "h"]
m2h = dict(zip(hom["m"].str.upper(), hom["h"].str.upper()))

gene_sets = {
    "Axolotl": sorted(set(ax["gene"].dropna().astype(str).str.upper())),
    "Mouse": sorted({m2h[g] for g in ms["gene"].dropna().astype(str).str.upper() if g in m2h}),
    "Human": sorted(set(hu["gene_symbol"].dropna().astype(str).str.upper())),
}

libs = ["GO_Biological_Process_2023", "KEGG_2021_Human", "Reactome_2022"]
sig_terms = {}
for name, genes in gene_sets.items():
    enr = gp.enrichr(gene_list=genes, gene_sets=libs, organism="human", outdir=None)
    df = enr.results
    df = df[df["Adjusted P-value"] < 0.05].sort_values("Adjusted P-value")
    df.to_csv(os.path.join(tables_dir, f"pathways_{name.lower()}_v2.csv"), index=False)
    sig_terms[name] = df.set_index("Term")
    print(f"{name}: {len(genes)} genes -> {len(df)} significant pathways")

ax_t, ms_t, hu_t = (set(sig_terms[k].index) for k in ["Axolotl", "Mouse", "Human"])
print(f"\nAxolotl & Mouse shared pathways: {len(ax_t & ms_t)}")
print(f"Axolotl & Human shared pathways: {len(ax_t & hu_t)}")
print(f"Mouse & Human shared pathways: {len(ms_t & hu_t)}")
print(f"All three shared pathways: {len(ax_t & ms_t & hu_t)}")

shared3 = sorted(ax_t & ms_t & hu_t)
print("\nPathways significant in all three species:")
for t in shared3:
    print(" -", t)

pairs = sorted(ax_t & ms_t)
print("\nTop axolotl & mouse shared pathways (up to 25):")
for t in pairs[:25]:
    print(" -", t)

shared_df = pd.DataFrame({
    "Term": sorted(ax_t | ms_t | hu_t),
})
shared_df["Axolotl"] = shared_df["Term"].isin(ax_t)
shared_df["Mouse"] = shared_df["Term"].isin(ms_t)
shared_df["Human"] = shared_df["Term"].isin(hu_t)
shared_df["Species_Count"] = shared_df[["Axolotl", "Mouse", "Human"]].sum(axis=1)
shared_df.sort_values("Species_Count", ascending=False).to_csv(
    os.path.join(tables_dir, "pathway_overlap_all_species_v2.csv"), index=False)

Axolotl: 1753 genes -> 696 significant pathways
Mouse: 149 genes -> 7 significant pathways
Human: 60 genes -> 207 significant pathways

Axolotl & Mouse shared pathways: 2
Axolotl & Human shared pathways: 11
Mouse & Human shared pathways: 0
All three shared pathways: 0

Pathways significant in all three species:

Top axolotl & mouse shared pathways (up to 25):
 - External Encapsulating Structure Organization (GO:0045229)
 - Extracellular Structure Organization (GO:0043062)
